In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
# TODO: Read PDF data, feed into Claude
messages=[]

with open("earth.pdf", "rb") as f:
    document_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    # Document Block
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": document_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": "Summarize the contents of the PDF document I just uploaded."
    }
])

chat(messages)

Message(id='msg_01GeNFoDU93eDiaNLaVxH1Ye', container=None, content=[TextBlock(citations=None, text='# Summary of Earth Wikipedia Article\n\nThis document is a Wikipedia article about Earth, covering its fundamental characteristics and natural history.\n\n## Key Points:\n\n**Basic Description:**\n- Earth is the third planet from the Sun and the only known astronomical object to harbor life\n- It\'s an ocean world with 70.8% of its surface covered by water\n- The remaining 29.2% is land, mostly in continental landmasses\n\n**Physical Characteristics:**\n- Mean radius: 6,371 km\n- Mass: 5.97 × 10²⁴ kg\n- Densest planet in the Solar System\n- Takes ~365.25 days to orbit the Sun\n- Rotates on its axis in ~23 hours 56 minutes\n- Axial tilt of 23.44° produces seasons\n- One natural satellite: the Moon\n\n**Atmosphere & Climate:**\n- Composition: 78% nitrogen, 21% oxygen\n- Average surface temperature: 14.76°C (58.57°F)\n- Dynamic atmosphere with greenhouse gases that maintain liquid water\n\n

## Get responses from CLaude with Citations

In [5]:

# TODO: Read PDF data with citations, feed into Claude
messages=[]

with open("earth.pdf", "rb") as f:
    document_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    # Document Block
    {
    "type": "document",
    "source": {
        "type": "base64",
        "media_type": "application/pdf",
        "data": document_bytes,
        },
    "title": "earth.pdf",
    "citations": { "enabled": True }      
    },
    # Text Block
    {
        "type": "text",
        "text": "How is earth atmosphere & oceans were formed."
    }
])

response = chat(messages)
print(response)

Message(id='msg_01DksxhPJy9JmMV88XkauraS', container=None, content=[TextBlock(citations=[CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n", document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationPageLocation(cited_text='[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n', document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text')], model='claude-sonnet-4-5-20250929', ro